# Без лишних вопросов о сохранении

Почему JupyterLab спрашивает «сохранить?» при закрытии статьи, которую никто не правил, и как от этого избавиться: две настройки Lab, что делает плагин и что остаётся сделать руками.

## Причина

Статьи — ноутбуки из Markdown, ядро им не нужно. Но при открытии Lab запускает ядро Python, и оно записывает в метаданные ноутбука `kernelspec` и `language_info`.
- В статье их нет или они другие — Lab считает статью изменённой и спрашивает при закрытии.
- Уже есть и совпадают с тем, что пишет ядро, — не спрашивает (проверено 2026-11-21 на `development/topic` и всех статьях `games/pokemon`).

Запомнить в статье «No Kernel» нельзя: в JupyterLab 4 нет ни такой настройки, ни такого значения в метаданных (по исходникам Lab). Поэтому решение — чтобы метаданные в статье сразу совпадали с ядром, а само ядро не висело после закрытия.

## Настройки JupyterLab

Файл `jupyter_data/config/lab/user-settings/@jupyterlab/notebook-extension/tracker.jupyterlab-settings` (в Lab: Settings → Settings Editor → Notebook):
```
{
    "autoStartDefaultKernel": true,
    "kernelShutdown": true
}
```

| Настройка | Значение | Зачем |
|---|---|---|
| `autoStartDefaultKernel` (Automatically Start Preferred Kernel) | `true` | без неё на каждой статье без выбранного ядра всплывает окно выбора ядра |
| `kernelShutdown` (Shut Down Kernel on Close) | `true` | ядро гасится при закрытии статьи — фоновые процессы Python не копятся (важно на телефоне) |

Автосохранение Lab (по умолчанию включено) на вопрос при правильных метаданных не влияет. Этот файл настроек есть в репозитории и переносится вместе с проектом (`development/mcp-kb/install#files`). После правки файла — обновить страницу Lab.

## Что делает плагин

Когда статью записывает MCP или индексатор, а `kernelspec` в ней нет (новая статья, пустые метаданные), плагин пишет то, что записало бы ядро: `kernelspec` ядра по умолчанию и `language_info`, который он на старте спрашивает у интерпретатора ядра. Такие статьи не спрашивают даже при первом открытии. Подробности — `development/mcp-kb/storage#format`.

Уже записанный `kernelspec` и сохранения из веба плагин не трогает никогда. В хешах секций и `version` метаданные ядра не учитываются, поэтому сохранение без правок не возвращает статью в очередь индексатора и не даёт `conflict`.

## Что сделать руками

- **Статья создана или сохранена в вебе с пустыми метаданными** — при первом открытии сохранить один раз (Ctrl+S), дальше закрывается молча. Текст от этого не меняется.
- **Сменился Python ядра** (другая машина, телефон с 3.12 вместо 3.11, обновление) — старые статьи спросят ещё раз: снова сохранить один раз. Плагин чужой `kernelspec` не переписывает.
- **Спрашивает снова и снова** — проверить настройки из секции выше и что статью не правит кто-то ещё (открыта во второй вкладке, пишет ИИ): изменение на диске Lab тоже показывает как вопрос.